# Compare the three TrOCR experiments — CER / WER & training history

Reads the results **directly from the three experiment notebooks' saved outputs** and compares them:

1. **Zero-shot (base TrOCR)** — `microsoft/trocr-base-handwritten`, no training
   (`zeroShotTrocrBaseUsingGoogleColab.ipynb`)
2. **Fine-tuned (base TrOCR)** — base model fine-tuned on our train set
   (`finetuneTrocrBaseUsingGoogleColab.ipynb`)
3. **Fine-tuned from Bentham** — `checkpoint-5750` fine-tuned on our train set
   (`finetuneBenthamCheckpointUsingGoogleColab.ipynb`)

All three share the same train / val / test split and the same test set (`test_set_line_crops/`), so
their **test CER / WER are directly comparable**.

**Produces:**
- overall test CER & WER bar chart,
- per-book CER & WER grouped bars,
- training-history curves (val CER / WER and loss per epoch) for the two fine-tunes,
- a summary table.

**No JSON-on-Drive round-trip and no re-running.** `extract_result()` reads each notebook's results
straight from its saved cell outputs — preferring a `RESULT_JSON` block if the notebook was run with
the emit cell, and otherwise parsing the printed `Overall TEST` line / per-book table / history. Just
place this notebook next to the three experiment notebooks (or point `NOTEBOOKS_DIR` at their folder)
and run it top to bottom.

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Locate the experiment notebooks & define the result reader

Point `NOTEBOOKS_DIR` at the folder containing the three experiment notebooks (default `"."` —
the same folder as this notebook). `extract_result()` reads each notebook's results from its
**saved cell outputs**, in two ways:

1. if the notebook was run with the emit cell, it parses the `RESULT_JSON_BEGIN` / `RESULT_JSON_END`
   block (exact float values, full training history including loss);
2. otherwise it **falls back to parsing the printed output** — the `Overall TEST — CER … WER … (n=…)`
   line, the per-book table, the validation block / history table.

Either way **no notebook needs to be re-run** as long as it was saved with its outputs.

In [ ]:
import re

# Folder holding the three experiment notebooks (this notebook lives alongside them).
NOTEBOOKS_DIR = "."

# Map each experiment to the notebook whose saved outputs carry its results.
NOTEBOOK_FILES = {
    'zero_shot_base':        'zeroShotTrocrBaseUsingGoogleColab.ipynb',
    'finetune_base':         'finetuneTrocrBaseUsingGoogleColab.ipynb',
    'finetune_from_bentham': 'finetuneBenthamCheckpointUsingGoogleColab.ipynb',
}

# Table labels (the short EXPERIMENTS labels below are only for plot axes).
LABELS = {
    'zero_shot_base':        'Zero-shot (base TrOCR)',
    'finetune_base':         'Fine-tuned (base TrOCR)',
    'finetune_from_bentham': 'Fine-tuned from Bentham (ckpt-5750)',
}

_BEGIN, _END = 'RESULT_JSON_BEGIN', 'RESULT_JSON_END'

def _cell_text(cell):
    """All stdout / text output of a single code cell, concatenated."""
    t = ''
    for o in cell.get('outputs', []):
        if o.get('output_type') == 'stream':
            t += ''.join(o.get('text', []))
        elif o.get('output_type') in ('execute_result', 'display_data'):
            d = o.get('data', {})
            if 'text/plain' in d:
                t += ''.join(d['text/plain']) + '\n'
    return t

def _parse_printed(text, key):
    """Fallback: rebuild a result dict from the human-readable printed output."""
    m = re.search(r'Overall TEST\s*[—-]+\s*CER:\s*([\d.]+)\s+WER:\s*([\d.]+)\s*\(n=(\d+)\)', text)
    if not m:
        return None
    res = {'experiment': key, 'label': LABELS.get(key, key),
           'test': {'cer': float(m.group(1)), 'wer': float(m.group(2)), 'n': int(m.group(3))}}

    # per-book rows: "B1   24   0.0978   0.2628"  (CER/WER require a decimal point,
    # so the integer-only split-summary table never matches).
    seen = {}
    for bm in re.finditer(r'^\s*(B\d)\s+(\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)\s*$', text, re.M):
        seen.setdefault(bm.group(1),
                        {'book': bm.group(1), 'cer': float(bm.group(3)), 'wer': float(bm.group(4))})
    res['per_book'] = [seen[b] for b in sorted(seen)]

    # validation-history table: "epoch  eval_cer  eval_wer" header + numeric rows
    hb = re.search(r'epoch\s+eval_cer\s+eval_wer\s*\n'
                   r'((?:\s*\d+\.\d+\s+\d+\.\d+\s+\d+\.\d+\s*\n?)+)', text)
    if hb:
        hist = []
        for row in hb.group(1).strip().splitlines():
            p = row.split()
            if len(p) == 3:
                hist.append({'epoch': float(p[0]), 'eval_cer': float(p[1]), 'eval_wer': float(p[2])})
        res['history'] = hist or None
        if hist:
            res['best_val_cer'] = min(h['eval_cer'] for h in hist)
    else:
        res['history'] = None

    # zero-shot prints a VAL block instead of a training history
    vm = re.search(r'on VAL[^\n]*\n\s*CER:\s*([\d.]+)\s*\n\s*WER:\s*([\d.]+)', text)
    if vm:
        res['val'] = {'cer': float(vm.group(1)), 'wer': float(vm.group(2))}
    return res

def extract_result(nb_path, key):
    """Read an experiment notebook's results straight from its saved cell outputs.

    Prefers a RESULT_JSON_BEGIN/END block (present if the notebook was run with the emit
    cell); otherwise reconstructs the result from the printed CER/WER/per-book output.
    Returns (result_dict, how) or (None, how) if neither is present."""
    nb = json.load(open(nb_path, encoding='utf-8'))
    texts = [_cell_text(c) for c in nb.get('cells', []) if c.get('cell_type') == 'code']

    # 1) preferred: a printed JSON block (last one wins, i.e. the most recent run)
    block = None
    for t in texts:
        if _BEGIN in t and _END in t:
            block = t.split(_BEGIN, 1)[1].split(_END, 1)[0]
    if block is not None:
        try:
            res = json.loads(block)
            res.setdefault('label', LABELS.get(key, key))
            return res, 'json-block'
        except json.JSONDecodeError:
            pass  # fall through to text parsing

    # 2) fallback: parse the human-readable printed output
    return _parse_printed('\n'.join(texts), key), 'printed-output'

In [ ]:
# Order + display labels for the experiments (controls plotting order and colours).
EXPERIMENTS = [
    ('zero_shot_base',        'Zero-shot\n(base)'),
    ('finetune_base',         'Fine-tuned\n(base)'),
    ('finetune_from_bentham', 'Fine-tuned\n(Bentham)'),
]

print('Notebooks dir:', os.path.abspath(NOTEBOOKS_DIR))
for key, _ in EXPERIMENTS:
    p = os.path.join(NOTEBOOKS_DIR, NOTEBOOK_FILES[key])
    print(('  found    ' if os.path.exists(p) else '  MISSING  ') + p)

# Read the results from the experiment notebooks

In [ ]:
results = {}
for key, _label in EXPERIMENTS:
    path = os.path.join(NOTEBOOKS_DIR, NOTEBOOK_FILES[key])
    if not os.path.exists(path):
        print(f'WARNING: notebook not found: {path}')
        continue
    res, how = extract_result(path, key)
    if res is None:
        print(f'WARNING: no results in {path} — run that notebook and save it with outputs.')
    else:
        results[key] = res
        print(f'loaded {key:22s} via {how}')

present = [(k, lbl) for k, lbl in EXPERIMENTS if k in results]
labels  = [lbl for _k, lbl in present]
keys    = [k for k, _lbl in present]
print(f'\nLoaded {len(keys)} experiments:', keys)

# Overall test/val summary table
summary = pd.DataFrame([{
    'experiment':   results[k].get('label', k),
    'test_CER':     results[k]['test']['cer'],
    'test_WER':     results[k]['test']['wer'],
    'test_lines':   results[k]['test']['n'],
    'best_val_CER': results[k].get('best_val_cer', (results[k].get('val') or {}).get('cer')),
} for k in keys])
summary.round(4)

# Overall test CER / WER comparison
Lower is better. The fine-tuned models should sit well below the zero-shot baseline.

In [ ]:
test_cer = [results[k]['test']['cer'] for k in keys]
test_wer = [results[k]['test']['wer'] for k in keys]

x = np.arange(len(keys))
w = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, test_cer, w, label='CER', color='#4C72B0')
b2 = ax.bar(x + w/2, test_wer, w, label='WER', color='#DD8452')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('error rate (lower is better)')
ax.set_title('Test-set CER / WER by experiment')
ax.legend()
for bars in (b1, b2):
    for b in bars:
        ax.annotate(f'{b.get_height():.3f}', (b.get_x() + b.get_width()/2, b.get_height()),
                    ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

# Per-book test CER / WER
Grouped bars — one cluster per book, one bar per experiment. Shows which books each model handles best.

In [ ]:
# Long-form per-book frame across experiments
rows = []
for k in keys:
    for r in results[k]['per_book']:
        rows.append({'experiment': results[k].get('label', k), 'book': r['book'],
                     'cer': r['cer'], 'wer': r['wer']})
pb = pd.DataFrame(rows)

for metric in ['cer', 'wer']:
    pivot = pb.pivot(index='book', columns='experiment', values=metric).sort_index()
    pivot = pivot[[results[k].get('label', k) for k in keys]]  # keep experiment order
    ax = pivot.plot(kind='bar', figsize=(11, 5))
    ax.set_ylabel(metric.upper()); ax.set_xlabel('book')
    ax.set_title(f'Per-book test {metric.upper()} by experiment')
    ax.legend(title='', fontsize=9)
    plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()

# Training history — validation CER / WER per epoch
Only the two fine-tuned runs have a training history. The dashed line marks the **zero-shot test CER**
as a reference baseline.

In [ ]:
def history_df(key):
    h = results.get(key, {}).get('history')
    if not h:
        return None
    df = pd.DataFrame(h)
    if 'eval_cer' not in df:
        return None
    # one row per epoch that has an eval metric
    return df[df['eval_cer'].notna()].sort_values('epoch')

ft_keys = [k for k in keys if history_df(k) is not None]
baseline_cer = results.get('zero_shot_base', {}).get('test', {}).get('cer')

if ft_keys:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for k in ft_keys:
        h = history_df(k); lbl = results[k].get('label', k).replace('\n', ' ')
        axes[0].plot(h['epoch'], h['eval_cer'], marker='o', label=lbl)
        axes[1].plot(h['epoch'], h['eval_wer'], marker='s', label=lbl)
    if baseline_cer is not None:
        axes[0].axhline(baseline_cer, ls='--', color='gray', label='zero-shot test CER')
    axes[0].set_title('Validation CER per epoch'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('CER')
    axes[1].set_title('Validation WER per epoch'); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('WER')
    axes[0].legend(fontsize=9); axes[1].legend(fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print('No training history found (run the fine-tune notebooks).')

# Training history — loss curves
Train loss (logged every few steps) and validation loss per epoch, for the two fine-tuned runs.

In [ ]:
if ft_keys:
    fig, axes = plt.subplots(1, len(ft_keys), figsize=(7 * len(ft_keys), 5), squeeze=False)
    for ax, k in zip(axes[0], ft_keys):
        df = pd.DataFrame(results[k]['history'])
        lbl = results[k].get('label', k).replace('\n', ' ')
        if 'loss' in df:
            tr = df[df['loss'].notna()]
            ax.plot(tr['epoch'], tr['loss'], label='train loss', color='#4C72B0', alpha=0.8)
        if 'eval_loss' in df:
            ev = df[df['eval_loss'].notna()]
            ax.plot(ev['epoch'], ev['eval_loss'], marker='o', label='val loss', color='#C44E52')
        ax.set_title(f'Loss — {lbl}'); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
        ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print('No training history found (run the fine-tune notebooks).')

# Summary table

In [ ]:
def pct_drop(base, val):
    return None if base in (None, 0) or val is None else round(100 * (base - val) / base, 1)

zs_cer = results.get('zero_shot_base', {}).get('test', {}).get('cer')
zs_wer = results.get('zero_shot_base', {}).get('test', {}).get('wer')
tbl = []
for k in keys:
    t = results[k]['test']
    tbl.append({
        'experiment':       results[k].get('label', k).replace('\n', ' '),
        'test CER':         round(t['cer'], 4),
        'test WER':         round(t['wer'], 4),
        'CER vs zero-shot': pct_drop(zs_cer, t['cer']),  # % relative reduction
        'WER vs zero-shot': pct_drop(zs_wer, t['wer']),
        'best val CER':     (round(results[k]['best_val_cer'], 4)
                             if results[k].get('best_val_cer') is not None else None),
        'test lines':       t['n'],
    })
summary_tbl = pd.DataFrame(tbl)
print(summary_tbl.to_string(index=False))
summary_tbl

# Image-style comparison figure (CER / WER bars + training history)

One row, three panels — matching the reference styling: CER bars, WER bars, and the training history
of the fine-tune that logged train loss (twin axis: **Loss** on the left, **Error Rate** on the right).
Colours follow matplotlib's default tab palette (blue / orange / green / …) and bar values are labelled
to 4 decimals. Works for however many experiments are currently loaded.

In [ ]:
# Colours match matplotlib's default tab palette (blue / orange / green / red), as in the reference image.
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# Short, stacked x-axis labels per experiment (letter + name), in the loaded order.
SHORT = {
    'zero_shot_base':        'Zero-shot\ntrocr-base',
    'finetune_base':         'Cook\nfine-tuned (base)',
    'finetune_from_bentham': 'Bentham\nfine-tuned',
}
letters = [chr(ord('A') + i) for i in range(len(keys))]
xlabels = [f"{letters[i]}\n{SHORT.get(k, results[k].get('label', k))}" for i, k in enumerate(keys)]
colors  = [PALETTE[i % len(PALETTE)] for i in range(len(keys))]

cer = [results[k]['test']['cer'] for k in keys]
wer = [results[k]['test']['wer'] for k in keys]

def _hist(k):
    h = results.get(k, {}).get('history')
    return pd.DataFrame(h) if h else None

# Training-history panel: show Experiment C (the 3rd experiment) when it has a history;
# otherwise fall back to any fine-tune that logged eval metrics.
def _has_hist(k):
    return _hist(k) is not None and 'eval_cer' in _hist(k).columns
hist_key = (keys[2] if len(keys) >= 3 and _has_hist(keys[2])
            else next((k for k in keys if _hist(k) is not None and 'loss' in _hist(k).columns), None)
            or next((k for k in keys if _has_hist(k)), None))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
n_exp = len(keys)
ctitle = {2:'Two', 3:'Three', 4:'Four', 5:'Five'}.get(n_exp, str(n_exp))

# --- Panel 1: CER bars ---
bars = axes[0].bar(range(n_exp), cer, color=colors)
axes[0].set_title(f'CER: {ctitle}-Experiment Comparison'); axes[0].set_ylabel('CER'); axes[0].set_ylim(0, 0.40)
axes[0].set_xticks(range(n_exp)); axes[0].set_xticklabels(xlabels)
for rect, v in zip(bars, cer):
    axes[0].annotate(f'{v:.4f}', (rect.get_x()+rect.get_width()/2, v), ha='center', va='bottom', fontsize=9)

# --- Panel 2: WER bars ---
bars = axes[1].bar(range(n_exp), wer, color=colors)
axes[1].set_title(f'WER: {ctitle}-Experiment Comparison'); axes[1].set_ylabel('WER'); axes[1].set_ylim(0, 0.8)
axes[1].set_xticks(range(n_exp)); axes[1].set_xticklabels(xlabels)
for rect, v in zip(bars, wer):
    axes[1].annotate(f'{v:.4f}', (rect.get_x()+rect.get_width()/2, v), ha='center', va='bottom', fontsize=9)

# --- Panel 3: training history (twin axis) for the chosen fine-tune ---
ax = axes[2]
if hist_key is not None:
    h = _hist(hist_key)
    lossrows = h[h['loss'].notna()] if 'loss' in h.columns else h.iloc[0:0]
    evalrows = h[h['eval_cer'].notna()] if 'eval_cer' in h.columns else h.iloc[0:0]
    ax2 = ax.twinx(); ax2.grid(False)
    handles = []
    if not lossrows.empty:
        l1, = ax.plot(lossrows['epoch'], lossrows['loss'], marker='o', color='#1f77b4', label='Train loss')
        handles.append(l1)
    if not evalrows.empty:
        l2, = ax2.plot(evalrows['epoch'], evalrows['eval_cer'], marker='s', color='#2ca02c', label='Val CER')
        l3, = ax2.plot(evalrows['epoch'], evalrows['eval_wer'], marker='^', ls='--', color='#ff7f0e', label='Val WER')
        handles += [l2, l3]
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color='#1f77b4'); ax2.set_ylabel('Error Rate')
    for _a in (ax, ax2):                       # extra empty space at the top for the legend/curves
        _lo, _hi = _a.get_ylim()
        _a.set_ylim(_lo, _hi + 0.45 * (_hi - _lo))
    letter = letters[keys.index(hist_key)]
    ax.set_title(f'Experiment {letter} \u2014 Training History')
    if handles:
        ax.legend(handles, [h_.get_label() for h_ in handles], fontsize=7)
else:
    ax.set_title('Training History')
    ax.text(0.5, 0.5, 'no training history available', ha='center', va='center', transform=ax.transAxes)

n_lines = results[keys[0]]['test']['n'] if keys else 0
plt.suptitle('Captain Cook Journal HTR \u2014 ' + ctitle + '-Experiment Comparison\n'
             '30 pages, B1\u2013B6, 3 contributors | Test: ' + str(n_lines) + ' lines across 6 pages',
             y=1.04)
plt.tight_layout()
plt.show()

# Variant: CER & WER in one panel + black/marker training history

A modified version of the figure above:
- **CER and WER share a single panel** as grouped (side-by-side) bars, coloured by *metric* (CER vs WER)
  so the colour is consistent across experiments.
- The **Experiment C training-history** panel uses **black for all three series**, distinguished only by
  **marker shape + line style** (noted in each legend label), so no colour is reused from the bar panel.

In [ ]:
# Self-contained variant (runs after the loader cells, using `results` / `keys`).
PALETTE_METRIC = {'CER': '#1f77b4', 'WER': '#ff7f0e'}   # colour by metric, consistent across experiments
SHORT = {
    'zero_shot_base':        'Zero-shot\ntrocr-base',
    'finetune_base':         'Cook\nfine-tuned (base)',
    'finetune_from_bentham': 'Bentham & Cook\ndouble fine-tuned',
}
letters = [chr(ord('A') + i) for i in range(len(keys))]
xlabels = [f"{letters[i]}\n{SHORT.get(k, results[k].get('label', k))}" for i, k in enumerate(keys)]
ctitle  = {2:'Two', 3:'Three', 4:'Four', 5:'Five'}.get(len(keys), str(len(keys)))

cer = [results[k]['test']['cer'] for k in keys]
wer = [results[k]['test']['wer'] for k in keys]

def _hist(k):
    h = results.get(k, {}).get('history')
    return pd.DataFrame(h) if h else None
def _has_hist(k):
    return _hist(k) is not None and 'eval_cer' in _hist(k).columns
# training-history panel = Experiment C (3rd experiment) when available
hist_key = (keys[2] if len(keys) >= 3 and _has_hist(keys[2])
            else next((k for k in keys if _has_hist(k)), None))

fig, (axb, axh) = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Panel 1: grouped CER / WER bars (same panel, side by side) ---
x = np.arange(len(keys)); bw = 0.38
b1 = axb.bar(x - bw/2, cer, bw, label='CER', color=PALETTE_METRIC['CER'])
b2 = axb.bar(x + bw/2, wer, bw, label='WER', color=PALETTE_METRIC['WER'])
for bars, vals in ((b1, cer), (b2, wer)):
    for rect, v in zip(bars, vals):
        axb.annotate(f'{v:.4f}', (rect.get_x()+rect.get_width()/2, v), ha='center', va='bottom', fontsize=8)
axb.set_xticks(x); axb.set_xticklabels(xlabels)
axb.set_ylabel('Error rate'); axb.set_ylim(0, 0.8)
axb.set_title(f'CER / WER: {ctitle}-Experiment Comparison')
axb.legend(fontsize=9)

# --- Panel 2: Experiment C history in BLACK, distinguished by marker + linestyle ---
ax = axh
if hist_key is not None:
    h = _hist(hist_key)
    lossrows = h[h['loss'].notna()] if 'loss' in h.columns else h.iloc[0:0]
    evalrows = h[h['eval_cer'].notna()] if 'eval_cer' in h.columns else h.iloc[0:0]
    ax2 = ax.twinx(); ax2.grid(False)
    handles = []
    if not lossrows.empty:
        l1, = ax.plot(lossrows['epoch'], lossrows['loss'],
                      color='black', marker='o', ls='-',  label='Train loss (o)')
        handles.append(l1)
    if not evalrows.empty:
        l2, = ax2.plot(evalrows['epoch'], evalrows['eval_cer'],
                       color='black', marker='s', ls='-.', label='Val CER (s)')
        l3, = ax2.plot(evalrows['epoch'], evalrows['eval_wer'],
                       color='black', marker='^', ls='--', label='Val WER (^)')
        handles += [l2, l3]
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax2.set_ylabel('Error Rate')
    for _a in (ax, ax2):                       # extra empty space at the top for the legend/curves
        _lo, _hi = _a.get_ylim()
        _a.set_ylim(_lo, _hi + 0.45 * (_hi - _lo))
    letter = letters[keys.index(hist_key)]
    ax.set_title(f'Experiment {letter} \u2014 Training History')
    if handles:
        ax.legend(handles, [h_.get_label() for h_ in handles], fontsize=7)
else:
    ax.set_title('Training History')
    ax.text(0.5, 0.5, 'no training history available', ha='center', va='center', transform=ax.transAxes)

n_lines = results[keys[0]]['test']['n'] if keys else 0
plt.suptitle('Captain Cook Journal HTR \u2014 ' + ctitle + '-Experiment Comparison\n'
             '30 pages, B1\u2013B6, 3 contributors | Test: ' + str(n_lines) + ' lines across 6 pages', y=1.04)
plt.tight_layout(); plt.show()